# RAG on English BSI Documents — Simple RAG vs. BM25 vs. Neo4j KG-RAG
### Benchmarked on the **English subset of BSI-HotpotQA** (83 multi-hop English questions, 105 gold paragraphs, all from the Smart Meter Gateway Protection Profile)

Same benchmark design as the German BSI-HotpotQA notebook, retargeted at the **English**
questions only, with the knowledge graph rebuilt end-to-end:

| | German run | English run (this notebook) |
|---|---|---|
| language | German | English |
| corpus | 163 paragraphs / 4 BSI-CS docs | 105 paragraphs / 1 doc (SMGW-PP) |
| questions | 199 | 83 |
| hop scope | intra- and cross-document | **all intra-document** (single source doc) |
| KG store | in-memory `networkx` | **Neo4j** (real graph DB, queried via Cypher) |
| KG extractor | Qwen2.5-1.5B | Qwen2.5-0.5B (quick local prototype — raise this first if KG underperforms) |

Because every query needs **two** distinct paragraphs, the headline metric is
`AllSupport@k` — the share of questions where *all* gold paragraphs land in the top-k.
Ordinary Recall@k gives half credit for finding one hop, which hides exactly the failure
mode this notebook is built to study.

**German data is intentionally left out of this run.** `bsi_paragraph_corpus.json` /
`bsi_hotpotqa_all.json` hold the German half of the same benchmark — point `QA_FILE` /
`CORP_FILE` at them (and swap `EXTRACT_SYS`/`EXTRACT_FEWSHOT` back to German) to evaluate
that half once the English pipeline is tuned.

## 0. Install dependencies

Run once.

In [ ]:
%pip install -q "langchain>=0.3" langchain-community langchain-huggingface langchain-neo4j langchain-ollama \
    sentence-transformers transformers accelerate faiss-cpu rank_bm25 neo4j \
    beir pandas matplotlib tqdm
print("If imports below fail, run the pip line above (remove the leading # ).")
print("Also requires a running Ollama daemon with the LLM_MODEL pulled, e.g.: ollama pull llama3:8b")

## 1. Configuration

Every knob lives here.

**Neo4j**: this notebook expects a running Neo4j instance. For a quick local prototype on macOS:
```bash
brew install neo4j
/opt/homebrew/opt/neo4j/bin/neo4j-admin dbms set-initial-password <your-password>
brew services start neo4j
```
Then set `NEO4J_PASSWORD` below (or export `NEO4J_URI`/`NEO4J_USER`/`NEO4J_PASSWORD` as env vars).
Browse the graph at http://localhost:7474 once it is loaded (Section 6).

In [ ]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- models: this run is English-only, so we use an English-specialised embedder ----
EMB_MODEL = "BAAI/bge-small-en-v1.5"
BGE_QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
LLM_MODEL = "llama3:8b"    # served by Ollama, not loaded via transformers. On this machine
                            # (16GB RAM, Apple Silicon, already under memory pressure) an 8B
                            # model in bf16 needs ~16GB just to materialise once -- both plain
                            # transformers and transformers+torchao quantization hit that same
                            # peak during from_pretrained() before any quantization kicks in.
                            # Ollama runs the GGUF-quantized weights directly (~4.7GB resident
                            # for the default Q4_0 llama3:8b) and never materialises full
                            # precision, so it's the only path that fits here. Requires
                            # `ollama pull llama3:8b` and a running `ollama serve`.
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

# ---- data: English subset only ----------------------------------------------
QA_FILE   = "english_data.json"
CORP_FILE = "bsi_english_paragraphs.json"

N_QUERIES        = 20        # trial run over a subset of English questions; None = all 83
TOP_K            = 10
KG_MAX_CHARS     = 900
KG_HOPS          = 1         # tried 2: over this graph's avg entity degree (~2.9), a 2-hop
                              # expansion touched most of the pool and flooded scores with
                              # noise -- AllSupport@2 went 0.10 -> 0.0. Back to 1.
KG_STRUCT_WEIGHT = 1.0       # tried 2.0 (same as a direct seed hit) to make 6b-3's
                              # co-referenced-with edges count more -- that also backfired
                              # (AllSupport@2 0.10 -> 0.05: it over-promotes hub structural
                              # IDs ahead of the real match, even at 1 hop). Best-known
                              # KG-RAG config is the structural layer (6b-3) + KG_HOPS=1 +
                              # NO extra weighting, i.e. this value == 1.0 (a no-op, so the
                              # Cypher/local scoring below reduces to the original uniform
                              # expanded-neighbour weight).
RRF_K            = 60

# ---- Neo4j (local prototype instance) ----------------------------------------
NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "bsi-rag-proto")

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
# NOTE: this only affects the embedder now (Section 4) -- the LLM (Section 6a) runs
# through Ollama, out-of-process, so it isn't pinned by this device choice.
LLM_DEVICE = "cpu" if DEVICE == "mps" else DEVICE
print(f"Device: {DEVICE} (embedder)  |  Embedder: {EMB_MODEL}  |  LLM: {LLM_MODEL} (via Ollama @ {OLLAMA_BASE_URL})  |  Neo4j: {NEO4J_URI}")

## 2. Load the English BSI-HotpotQA subset

`bsi_english_paragraphs.json` = the 105 retrievable English paragraphs (all from the
Smart Meter Gateway Protection Profile, SMGW-PP).
`english_data.json` = the 83 English HotpotQA-schema questions; `supporting_facts` gives
the gold paragraph titles, which are the qrels.

In [ ]:
import json

corpus_units = json.load(open(CORP_FILE))
data         = json.load(open(QA_FILE))

assert all(it["metadata"]["language"] == "en" for it in data), "expected only English items in english_data.json"

corpus = {u["title"]: {"title": u["title"], "text": " ".join(u["sentences"])}
          for u in corpus_units}
queries = {it["_id"]: it["question"] for it in data}
qrels   = {it["_id"]: {t: 1 for t, _ in it["supporting_facts"]} for it in data}
meta    = {it["_id"]: it for it in data}

n_q = N_QUERIES or len(queries)
print(f"corpus paragraphs   : {len(corpus):,}")
print(f"questions (English) : {len(queries):,}")
print(f"gold per question   : {np.mean([len(v) for v in qrels.values()]):.2f}")

qid = next(iter(queries))
print("\nExample query:", queries[qid])
print("Gold paragraphs:", list(qrels[qid]))

## 3. Document universe

The corpus is only 105 paragraphs (single document), so no BM25 pooling step is needed —
**every system retrieves over the entire corpus**. The recall ceiling is a true 1.0 for
all systems.

In [ ]:
from rank_bm25 import BM25Okapi
import re

def make_text(d): return (d.get("title","") + ". " + d.get("text","")).strip()

def tokenize(s):
    return re.findall(r"\w+", s.lower(), flags=re.UNICODE)

all_doc_ids = list(corpus)
eval_qids   = list(queries)[:n_q]

if N_QUERIES is not None and N_QUERIES < len(queries):
    # Trial run: Section 6b's LLM extraction cost scales with pool size, not query count,
    # so subsetting eval_qids alone wouldn't actually speed anything up. Shrink the pool to
    # the trial questions' gold paragraphs plus a few distractors (same trick as Section 12)
    # so extraction only pays for what this trial touches, while keeping the recall ceiling
    # at 1.0 for all systems.
    _trial_rng = random.Random(SEED)
    trial_gold = {d for q in eval_qids for d in qrels[q]}
    remaining  = [d for d in all_doc_ids if d not in trial_gold]
    n_distractors = min(20, len(remaining))
    pool_ids = sorted(trial_gold) + _trial_rng.sample(remaining, n_distractors)
else:
    pool_ids = all_doc_ids                     # whole corpus

pool_texts = {d: make_text(corpus[d]) for d in pool_ids}
eval_qrels = {q: qrels[q] for q in eval_qids}

print(f"Evaluating {len(eval_qids)} English queries over a pool of {len(pool_ids)} paragraphs.")

## 4. System 1 — Simple RAG (dense retrieval)

LangChain `HuggingFaceEmbeddings` + `FAISS`. This is the classic "embed everything,
nearest-neighbour search" RAG retriever.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name=EMB_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

def as_passage(t): return t
def as_query(t):   return BGE_QUERY_INSTRUCTION + t

docs = [Document(page_content=as_passage(pool_texts[d]), metadata={"doc_id": d})
        for d in pool_ids]
vectorstore = FAISS.from_documents(docs, embeddings)

def dense_search(query, k=TOP_K):
    hits = vectorstore.similarity_search_with_score(as_query(query), k=k)
    return {h.metadata["doc_id"]: float(-dist) for h, dist in hits}

dense_run = {q: dense_search(queries[q]) for q in eval_qids}
print("Dense retriever ready. Example top-3:",
      list(dense_run[eval_qids[0]].items())[:3])

## 5. System component — BM25 (lexical)

`rank_bm25` directly, scored. Used both standalone and inside the hybrid.

In [ ]:
_bm25 = BM25Okapi([tokenize(pool_texts[d]) for d in pool_ids])

def bm25_search(query, k=TOP_K):
    scores = _bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]
    return {pool_ids[i]: float(scores[i]) for i in order}

bm25_run = {q: bm25_search(queries[q]) for q in eval_qids}
print("BM25 retriever ready. Example top-3:",
      list(bm25_run[eval_qids[0]].items())[:3])

## 6. System 2 — Knowledge-Graph RAG, backed by Neo4j

### 6a. Load the LLM through Ollama
We use `ChatOllama`, which talks to a local Ollama daemon over HTTP and applies the
model's chat template automatically — same `SystemMessage`/`HumanMessage` interface as
the previous `ChatHuggingFace` setup, so nothing downstream (6b, 6d) needs to change.
Progression on this benchmark: `Qwen2.5-0.5B-Instruct` extracted only 2.5 triples/paragraph
and scored `AllSupport@2 = 0.0`; `Qwen2.5-1.5B-Instruct` and then `Qwen2.5-3B-Instruct`
improved on that. This run swaps in **`llama3:8b`** (served by Ollama, Q4_0 GGUF, ~4.7GB
resident) to see whether a larger, differently-trained model extracts cleaner/denser
triples than the Qwen family did at this parameter count.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

chat = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)

def llm(system, user):
    return chat.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content

print("LLM ready:", llm("You are terse.", "Reply with the single word: ok"))

### 6b. Extract (head | relation | tail) triples with the LLM

Small models handle a `pipe`-delimited format far more reliably than JSON, so we use that
and parse tolerantly. Extraction runs **once per pooled document**.

In [ ]:
from tqdm.auto import tqdm

EXTRACT_SYS = (
    "You extract facts from English IT-security texts. "
    "Output ONLY lines in the format: head | relation | tail . "
    "Short noun phrases, no comments, no numbering."
)
EXTRACT_FEWSHOT = (
    "Text: The firewall objective O.Firewall protects the LMN and HAN from threats "
    "originating in the WAN and blocks unauthorised connections.\n"
    "o.firewall | protects | lmn and han\n"
    "o.firewall | blocks | unauthorised connections\n\n"
    "Text: {doc}\n"
)

def parse_triples(text):
    triples = []
    for line in text.splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) == 3 and all(parts) and len(parts[0]) < 80 and len(parts[2]) < 80:
            triples.append((parts[0].lower(), parts[1].lower(), parts[2].lower()))
    return triples

def extract_triples(doc_text):
    try:
        return parse_triples(llm(EXTRACT_SYS, EXTRACT_FEWSHOT.format(doc=doc_text[:KG_MAX_CHARS])))
    except Exception:
        return []

doc_triples = {}
for d in tqdm(pool_ids, desc="LLM KG extraction"):
    doc_triples[d] = extract_triples(pool_texts[d])

n_triples = sum(len(v) for v in doc_triples.values())
print(f"Extracted {n_triples} triples from {len(pool_ids)} paragraphs "
      f"({n_triples/max(1,len(pool_ids)):.1f} per paragraph).")

### 6b-3. Hierarchical structural layer — deterministic ID cross-references

The LLM triples above are extracted **per paragraph, in isolation**, so they can only
capture what a small model notices in ~900 characters of text — they have no way to
know that the paragraph they're reading is the *counter* to a threat defined three
sections away. But this corpus already encodes that structure explicitly: every
Threat/Objective/Assumption gets a stable ID (`T.x`, `O.x`, `OE.x`, `A.x`) in its own
section title, and — critically — **"Countering T.x" paragraphs name the objective IDs
that counter them directly in the text** (e.g. *"T.DataModificationWAN is countered by
... O.Firewall and O.Crypt"*), which is exactly the gold-paragraph pair in this
benchmark's worked example.

This adds a second, higher-level graph layer built with regex instead of an LLM: extract
every `T./O./OE./A.` ID mentioned in each paragraph and add a `co-referenced-with` edge
between every pair mentioned together. It's deterministic (no extraction noise), cheap
(no LLM calls), and merged into the *same* `doc_triples` dict the LLM populated above, so
it flows through entity resolution (6b-2) and the Neo4j load (6c) unchanged — the
hierarchy is "structural IDs sit above free-text entities, and both layers share nodes
where the surface forms match."

In [ ]:
ID_PATTERN = re.compile(r"\b(?:OE|[TOA])\.[A-Za-z][A-Za-z0-9]*\b")

def extract_ids(text):
    return sorted({m.lower() for m in ID_PATTERN.findall(text)})

n_structural, n_docs_with_ids = 0, 0
for d in pool_ids:
    ids = extract_ids(pool_texts[d])
    if ids:
        n_docs_with_ids += 1
    pairs = [(a, "co-referenced-with", b) for i, a in enumerate(ids) for b in ids[i + 1:]]
    doc_triples.setdefault(d, []).extend(pairs)
    n_structural += len(pairs)

print(f"Structural layer: {n_structural} deterministic co-reference triples added from "
      f"regex-matched section IDs ({n_docs_with_ids}/{len(pool_ids)} paragraphs contain at least one).")
print("Example (the benchmark's worked bridge case):",
      extract_ids(pool_texts.get("SMGW-PP – 4.3.2 Countering T.DataModificationWAN", "")))

### 6b-2. Resolve entities across paragraphs

Extraction in 6b runs **once per paragraph, independently**, so the same real-world entity
gets a different surface string in every paragraph that mentions it — e.g. `"o.firewall"`
in one paragraph, `"the firewall objective"` in another. `MERGE (e:Entity {name})` in 6c
then matches on that raw string, so instead of one connected graph we get ~105
disconnected star-shapes, one per paragraph: the run that produced the numbers in Section
8 loaded 371 triples into 378 entities — barely 2 mentions per entity on average, i.e.
almost no entity is shared between two paragraphs. Since a 2-hop bridge question needs
exactly that kind of shared entity to walk from gold-paragraph-1 to gold-paragraph-2 in
the graph, this fragmentation is the most likely cause of `AllSupport@2 = 0.024` —
worse than doing nothing.

This clusters entity strings by embedding cosine similarity (reusing the `EMB_MODEL`
embedder already loaded in Section 4) and rewrites every triple to use one canonical name
per cluster, *before* anything is loaded into Neo4j — so the graph gains real
cross-paragraph edges instead of one island per paragraph.

In [ ]:
from collections import defaultdict

ENTITY_SIM_THRESHOLD = 0.86  # cosine similarity above which two entity strings are merged into one node

def resolve_entities(triples_by_doc, sim_threshold=ENTITY_SIM_THRESHOLD):
    """Cluster entity strings by embedding similarity and rewrite triples to use one
    canonical name per cluster. Returns (resolved_triples_by_doc, raw_name -> canonical_name)."""
    ents = sorted({h for t in triples_by_doc.values() for h, _, _ in t} |
                  {tt for t in triples_by_doc.values() for _, _, tt in t})
    if not ents:
        return dict(triples_by_doc), {}

    vecs = np.array(embeddings.embed_documents(ents))
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0   # guard: a degenerate (all-zero) embedding would otherwise NaN out via 0/0
    vecs = vecs / norms
    sim = vecs @ vecs.T

    parent = list(range(len(ents)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj: parent[ri] = rj

    for i in range(len(ents)):
        for j in range(i + 1, len(ents)):
            if sim[i, j] >= sim_threshold:
                union(i, j)

    clusters = defaultdict(list)
    for i in range(len(ents)):
        clusters[find(i)].append(i)

    canon = {}
    for members in clusters.values():
        rep = min((ents[i] for i in members), key=len)   # shortest surface form as canonical name
        for i in members:
            canon[ents[i]] = rep

    resolved = {d: [(canon[h], r, canon[t]) for h, r, t in triples]
                for d, triples in triples_by_doc.items()}
    return resolved, canon

doc_triples_resolved, entity_canon = resolve_entities(doc_triples)

n_raw      = len(set(entity_canon))
n_resolved = len(set(entity_canon.values()))
merged_examples = defaultdict(list)
for raw, rep in entity_canon.items():
    if raw != rep:
        merged_examples[rep].append(raw)

print(f"Entity resolution: {n_raw} raw entity strings -> {n_resolved} canonical entities "
      f"({n_raw - n_resolved} merged, cosine >= {ENTITY_SIM_THRESHOLD}).")
for rep, raws in list(merged_examples.items())[:8]:
    print(f"  {rep!r:35s} <- {raws}")

### 6c. Load the graph into Neo4j

Schema:
- `(:Entity {name})` nodes, one per extracted head/tail phrase
- `(:Entity)-[:REL {type}]->(:Entity)` for each extracted triple
- `(:Entity)-[:MENTIONED_IN]->(:Document {id})` for provenance — **this is what lets the
  graph retrieve documents**, exactly as `node_docs` did in the in-memory version, just
  persisted in Neo4j instead of a Python dict.

The whole load is one parameterised `UNWIND` + `MERGE` Cypher statement instead of a
Python loop — this is where the "search the nodes and queries in Neo4j" part starts.

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASSWORD,
                    refresh_schema=False)  # schema introspection needs the APOC plugin; we only use graph.query()

graph.query("MATCH (n) DETACH DELETE n")  # fresh graph for this benchmark run
graph.query("CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
graph.query("CREATE CONSTRAINT doc_id IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE")

rows = [{"doc": d, "head": h, "relation": r, "tail": t}
        for d, triples in doc_triples_resolved.items() for h, r, t in triples]

graph.query(
    """
    UNWIND $rows AS row
    MERGE (h:Entity {name: row.head})
    MERGE (t:Entity {name: row.tail})
    MERGE (h)-[:REL {type: row.relation}]->(t)
    MERGE (doc:Document {id: row.doc})
    MERGE (h)-[:MENTIONED_IN]->(doc)
    MERGE (t)-[:MENTIONED_IN]->(doc)
    """,
    params={"rows": rows},
)

n_entities = graph.query("MATCH (e:Entity) RETURN count(e) AS n")[0]["n"]
n_rels     = graph.query("MATCH ()-[r:REL]->() RETURN count(r) AS n")[0]["n"]
print(f"Neo4j knowledge graph: {n_entities} entities, {n_rels} relations, "
      f"loaded from {len(rows)} triples (entity-resolved). Browse at http://localhost:7474")

### 6d. Graph-based retrieval — a Cypher query per question

For each query we (1) pull candidate entities from the query (LLM extractor + token
fallback, same as before), then (2) send those candidates straight into Neo4j as query
parameters. **The matching, hop expansion, and document scoring all happen inside Neo4j**
via one Cypher statement: seed entities score 2.0, and expanded neighbours (up to
`KG_HOPS` hops) score `KG_STRUCT_WEIGHT` if *any* hop on the path used a 6b-3
`co-referenced-with` structural edge, else 1.0 for a purely LLM-guessed edge — so the
graph leans on the deterministic layer over the noisier LLM one whenever a path can use
it, the same scoring rule as the original `networkx` version but now with a
structural-vs-freeform distinction, executed as a graph-database query instead of a
Python BFS.

In [ ]:
def query_entities(query):
    ents = set()
    for h, _, t in extract_triples(query):     # reuse the LLM extractor on the query
        ents.add(h); ents.add(t)
    if not ents:
        # lexical fallback only when the LLM found nothing — mixing in generic query
        # tokens alongside real entities was diluting seeds with substring hits on
        # unrelated documents (short common words match too many entity phrases).
        ents.update(w for w in tokenize(query) if len(w) > 4)
    return list(ents)

# Seeds score 2.0 (direct hit). Expanded neighbours score KG_STRUCT_WEIGHT if ANY hop on
# the path used a 6b-3 `co-referenced-with` edge (regex-exact structural link), else 1.0
# for a purely LLM-guessed edge — so the graph leans on the reliable layer when it's
# available, same idea as boosting a high-precision signal in a fused ranker.
CYPHER_KG_SEARCH = f"""
UNWIND $seeds AS s
MATCH (seed:Entity)
WHERE toLower(seed.name) CONTAINS s OR s CONTAINS toLower(seed.name)
WITH collect(DISTINCT seed) AS seedNodes

UNWIND seedNodes AS sn
OPTIONAL MATCH p = (sn)-[:REL*0..{KG_HOPS}]-(nbr:Entity)
WITH seedNodes, nbr, p
WHERE nbr IS NOT NULL AND NOT nbr IN seedNodes
WITH seedNodes, nbr,
     max(CASE WHEN any(r IN relationships(p) WHERE r.type = 'co-referenced-with')
              THEN {KG_STRUCT_WEIGHT} ELSE 1.0 END) AS w
WITH seedNodes, collect({{node: nbr, w: w}}) AS expanded

UNWIND ([sd IN seedNodes | {{node: sd, w: 2.0}}] + expanded) AS item
WITH item.node AS n, item.w AS w
MATCH (n)-[:MENTIONED_IN]->(doc:Document)
WITH doc, sum(w) AS score
RETURN doc.id AS doc_id, score
ORDER BY score DESC
LIMIT $k
"""

def kg_search(query, k=TOP_K):
    seeds = query_entities(query)
    if not seeds:
        return {}
    rows = graph.query(CYPHER_KG_SEARCH, params={"seeds": seeds, "k": k})
    return {r["doc_id"]: r["score"] for r in rows}

kg_run = {q: kg_search(queries[q]) for q in eval_qids}
_covered = sum(1 for q in eval_qids if kg_run[q])
print(f"Neo4j KG retriever ready. Returned results for {_covered}/{len(eval_qids)} queries.")
print("Example top-3:", list(kg_run[eval_qids[0]].items())[:3])

## 7. System 3 — Hybrid (Dense + BM25 + KG) via Reciprocal Rank Fusion

RRF combines rankings without needing comparable score scales: a doc at rank *r* in a
list contributes `1 / (RRF_K + r)`. Robust and parameter-light.

In [ ]:
from collections import defaultdict

def rrf_fuse(runs, k=RRF_K, topk=TOP_K):
    fused = {}
    for q in eval_qids:
        agg = defaultdict(float)
        for run in runs:
            ranked = sorted(run.get(q, {}).items(), key=lambda x: -x[1])
            for rank, (doc_id, _) in enumerate(ranked):
                agg[doc_id] += 1.0 / (k + rank + 1)
        fused[q] = dict(sorted(agg.items(), key=lambda x: -x[1])[:topk])
    return fused

hybrid_run = rrf_fuse([dense_run, bm25_run, kg_run])
print("Hybrid (Dense+BM25+KG) ready. Example top-3:",
      list(hybrid_run[eval_qids[0]].items())[:3])

## 8. Evaluation

BEIR's `pytrec_eval` metrics, **plus `AllSupport@k`** — the fraction of questions whose
*complete* supporting set is inside the top-k. For a 2-hop dataset this is the metric
that matters: Recall@10 rewards a system that reliably nails hop 1 and never finds hop 2.

In [ ]:
from beir.retrieval.evaluation import EvaluateRetrieval
import pandas as pd

evaluator = EvaluateRetrieval()
K = [2, TOP_K]

def all_support(run, k):
    hits = []
    for q, gold in eval_qrels.items():
        top = sorted(run.get(q, {}).items(), key=lambda x: -x[1])[:k]
        hits.append(float(set(gold) <= {d for d, _ in top}))
    return float(np.mean(hits))

def score(run):
    ndcg, _map, recall, precision = evaluator.evaluate(eval_qrels, run, K)
    return {
        f"nDCG@{TOP_K}":    ndcg[f"NDCG@{TOP_K}"],
        f"Recall@{TOP_K}":  recall[f"Recall@{TOP_K}"],
        "MAP":              _map[f"MAP@{TOP_K}"],
        f"P@{TOP_K}":       precision[f"P@{TOP_K}"],
        "AllSupport@2":     all_support(run, 2),
        f"AllSupport@{TOP_K}": all_support(run, TOP_K),
    }

systems = {
    "Simple RAG (Dense)":        dense_run,
    "BM25 (lexical)":            bm25_run,
    "KG-RAG (Neo4j + LLM graph)": kg_run,
    "Hybrid (Dense+BM25+KG)":     hybrid_run,
}
results = pd.DataFrame({name: score(run) for name, run in systems.items()}).T.round(4)
results

In [ ]:
import matplotlib.pyplot as plt

ax = results[[f"nDCG@{TOP_K}", f"Recall@{TOP_K}", "AllSupport@2", f"AllSupport@{TOP_K}"]] \
        .plot(kind="bar", figsize=(10, 5), rot=15)
ax.set_title(f"BSI-HotpotQA (English) — {len(eval_qids)} multi-hop questions, {len(pool_ids)} paragraphs")
ax.set_ylabel("score"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

best = results["AllSupport@2"].idxmax()
print(f"Best AllSupport@2: {best}  ({results.loc[best, 'AllSupport@2']:.4f})")

## 9. Where the systems actually differ

Aggregate scores hide the multi-hop story. All English questions are `intra-document`
(single source doc), so unlike the German run there is no cross- vs. intra-document
split to look at — instead we slice by question type/difficulty, and look at the
per-hop rank gap (how much deeper the *second* gold paragraph sits than the first).

In [ ]:
# --- slice analysis ---------------------------------------------------------
def slice_scores(run, qs):
    global eval_qrels
    keep, eval_qrels = eval_qrels, {q: qrels[q] for q in qs}
    try:    out = score({q: run[q] for q in qs})
    finally: eval_qrels = keep
    return out

def sel(key, val):
    return [q for q in eval_qids
            if (meta[q].get(key) or meta[q]["metadata"].get(key)) == val]

rows = {}
for key, val in [("type","bridge"), ("type","comparison"),
                  ("level","easy"), ("level","medium"), ("level","hard")]:
    qs = sel(key, val)
    if qs:
        rows[f"{key}={val} (n={len(qs)})"] = {
            n: round(slice_scores(r, qs)[f"AllSupport@{TOP_K}"], 3)
            for n, r in systems.items()}
display(pd.DataFrame(rows).T)

# --- per-hop rank gap -------------------------------------------------------
full = {q: dense_search(queries[q], k=len(pool_ids)) for q in eval_qids}
first, last = [], []
for q in eval_qids:
    order = [d for d, _ in sorted(full[q].items(), key=lambda x: -x[1])]
    rk = sorted(order.index(g) + 1 for g in eval_qrels[q])
    first.append(rk[0]); last.append(rk[-1])
print(f"median rank, easiest gold paragraph: {np.median(first):.0f}")
print(f"median rank, hardest gold paragraph: {np.median(last):.0f}   <-- the second hop")

# --- one query, four retrievers ---------------------------------------------
q = eval_qids[0]
gold = set(qrels.get(q, {}))
print("\nQUERY:", queries[q]); print("GOLD :", gold, "\n")
for name, run in systems.items():
    top = list(run.get(q, {}))[:5]
    print(f"{name:28s} -> {[d[:42] + ('  OK' if d in gold else '') for d in top]}")

## 10. Next steps

- **Raise `LLM_MODEL`** (Qwen2.5-1.5B/7B-Instruct or an API model) if Section 6b's triples
  look sparse or noisy for a question class — this is the single biggest lever on
  KG-RAG quality and was the known weak point of the local-0.5B prototype.
- **Evaluate the German half.** Point `QA_FILE`/`CORP_FILE` at `bsi_hotpotqa_all.json` /
  `bsi_paragraph_corpus.json`, switch `EXTRACT_SYS`/`EXTRACT_FEWSHOT` back to German, and
  re-run — the Neo4j graph and Cypher retrieval code do not need to change, only the
  extraction prompt and the loaded data.
- **Tune `kg_search`.** `KG_HOPS`, the seed/expanded weighting (currently 2.0 / 1.0), and
  the substring-match seeding are all easy Cypher-side knobs — inspect the graph at
  http://localhost:7474 to see why a given query's seeds under- or over-match.
- **Add more BSI documents** to `bsi_english_paragraphs.json` to see how each retriever
  degrades as the haystack grows past a single protection profile.

## 12. Quick subset check — does entity resolution actually help?

A full re-run of Sections 6–8 costs a fresh LLM extraction pass plus a Neo4j reload.
Before paying that cost, this section gets a fast read on whether `resolve_entities()`
(6b-2) is worth it: it builds a small, self-contained sub-corpus (a handful of bridge
questions' gold paragraphs plus a few distractors), extracts triples on just that subset,
and compares **raw** vs **entity-resolved** `AllSupport@2/@{TOP_K}` using a pure-Python
BFS scorer (`kg_search_local`) that mirrors `CYPHER_KG_SEARCH`'s logic exactly — seed
weight 2.0, `KG_STRUCT_WEIGHT` for any path that touches a `co-referenced-with` edge,
1.0 otherwise, same `KG_HOPS` — so no Neo4j round-trip is needed for this sanity check.
Re-run 6–9 for the full-scale number once this looks promising.

In [ ]:
import random as _random

_rng = _random.Random(SEED)

N_TEST_QUERIES     = 12
N_TEST_DISTRACTORS = 10

bridge_qs = [q for q in eval_qids if (meta[q].get("type") or meta[q]["metadata"].get("type")) == "bridge"]
test_qids = _rng.sample(bridge_qs, min(N_TEST_QUERIES, len(bridge_qs)))

test_gold_docs = {d for q in test_qids for d in qrels[q]}
remaining      = [d for d in pool_ids if d not in test_gold_docs]
test_pool      = sorted(test_gold_docs) + _rng.sample(remaining, min(N_TEST_DISTRACTORS, len(remaining)))

print(f"Quick subset: {len(test_qids)} bridge questions, {len(test_pool)} paragraphs "
      f"({len(test_gold_docs)} gold + {len(test_pool) - len(test_gold_docs)} distractors).")

test_triples = {d: extract_triples(pool_texts[d]) for d in tqdm(test_pool, desc="subset extraction")}
n_test_triples = sum(len(v) for v in test_triples.values())
print(f"Extracted {n_test_triples} triples from {len(test_pool)} paragraphs "
      f"({n_test_triples / max(1, len(test_pool)):.1f} per paragraph).")

In [ ]:
def build_adjacency(triples_by_doc):
    adj, adj_struct, ent_docs = defaultdict(set), defaultdict(set), defaultdict(set)
    for d, triples in triples_by_doc.items():
        for h, r, t in triples:
            adj[h].add(t); adj[t].add(h)
            if r == "co-referenced-with":
                adj_struct[h].add(t); adj_struct[t].add(h)
            ent_docs[h].add(d); ent_docs[t].add(d)
    return adj, adj_struct, ent_docs

def kg_search_local(triples_by_doc, query, hops=KG_HOPS, k=TOP_K, struct_weight=KG_STRUCT_WEIGHT):
    """Pure-Python mirror of CYPHER_KG_SEARCH: substring seed match, undirected hop
    expansion, seeds weighted 2x, expanded neighbours weighted struct_weight if reached
    via any co-referenced-with edge on the path (else 1.0)."""
    adj, adj_struct, ent_docs = build_adjacency(triples_by_doc)
    seeds = query_entities(query)
    matched = {e for e in adj for s in seeds if s in e or e in s}

    struct_reach = defaultdict(bool)
    visited = set(matched)
    frontier = set(matched)
    for _ in range(hops):
        nxt = set()
        for e in frontier:
            for nb in adj[e]:
                if nb in adj_struct[e] or struct_reach[e]:
                    struct_reach[nb] = True
                if nb not in visited:
                    nxt.add(nb)
        visited |= nxt
        frontier = nxt
    expanded = visited - matched

    scores = defaultdict(float)
    for e in matched:
        for d in ent_docs.get(e, ()):
            scores[d] += 2.0
    for e in expanded:
        w = struct_weight if struct_reach[e] else 1.0
        for d in ent_docs.get(e, ()):
            scores[d] += w
    return dict(sorted(scores.items(), key=lambda x: -x[1])[:k])

def all_support_local(run, qs, k):
    hits = [float(set(qrels[q]) <= {d for d, _ in sorted(run.get(q, {}).items(), key=lambda x: -x[1])[:k]})
            for q in qs]
    return float(np.mean(hits))

raw_run           = {q: kg_search_local(test_triples, queries[q]) for q in test_qids}
resolved_triples, _ = resolve_entities(test_triples)
resolved_run      = {q: kg_search_local(resolved_triples, queries[q]) for q in test_qids}

comparison = pd.DataFrame({
    "raw (no entity resolution)": {
        "AllSupport@2":        all_support_local(raw_run, test_qids, 2),
        f"AllSupport@{TOP_K}": all_support_local(raw_run, test_qids, TOP_K),
    },
    "resolved (Section 6b-2 fix)": {
        "AllSupport@2":        all_support_local(resolved_run, test_qids, 2),
        f"AllSupport@{TOP_K}": all_support_local(resolved_run, test_qids, TOP_K),
    },
}).T
comparison